# 📊 Github Code Analyst

## ⚙️ 설정
- 필요 SDK 설치
- 필요 라이브러리 불러오기
- 환경변수 가져오기

In [ ]:
# %pip install --upgrade pip # 필요 시
PYGITHUB_VERSION = "2.8.1"

%pip install PyGithub=={PYGITHUB_VERSION}

In [ ]:
# Import
import asyncio
import os
import logging
import json
import shutil

from pathlib import Path
from datetime import datetime
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import List

from github import Github
from github import GithubException
from github.Auth import Token
from github.Repository import Repository

from dotenv import load_dotenv


# 초기 설정

## env
load_dotenv(override=True)

## logger
logger_level = logging.INFO
formatter = logging.Formatter('%(asctime)s - %(levelname)s - [%(name)s] %(message)s')

logger = logging.getLogger("GithubCodeAnalyst")
logger.setLevel(logger_level)

if logger.hasHandlers():
    logger.handlers.clear()
stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)


In [ ]:
# Abstract
class _GithubLoginRequest(ABC):
    """
    Github 로그인을 위한 인증 요청 DTO
    """
    pass

class _GithubAuthenticator(ABC):
    @abstractmethod
    def githubLogin(self, request: _GithubLoginRequest) -> Github:
        pass
    
# Implementation
@dataclass
class GithubTokenLoginRequest(_GithubLoginRequest):
    """
    Github 개인 토큰을 이용한 로그인 요청 DTO
    """
    token: str
    def __post_init__(self):
        if not self.token or not self.token.strip():
            raise ValueError("GitHub 토큰은 비어있거나 공백일 수 없습니다.")

class GithubTokenAuthenticator(_GithubAuthenticator):
    def githubLogin(self, request: GithubTokenLoginRequest) -> Github:
        """
        Github 개인 토큰을 통해 로그인 및 Github 객체 생성

    
        Args:
            request (GithubTokenLoginRequest): GitHub 토큰이 포함된 DTO 객체.
        Returns:
            Github: 인증이 완료되어 API 통신에 바로 사용할 수 있는 PyGithub 클라이언트 객체.
        Raises:
            Exception: 토큰이 유효하지 않거나, GitHub API 서버와 통신할 수 없는 경우 발생합니다. 
        """
        github_token = Token(request.token)
        github = Github(auth=github_token)
        try:
            user = github.get_user()
            logger.info("[GITHUB_LOGIN] Success to login | username : " + user.login)
            return github
        except Exception as e:
            logger.error(f"[GITHUB_LOGIN] Fail to authenticate user | {e}", exc_info=True)
            raise

In [ ]:
# 로그인 전략 선택 및 객체 생성

## 토큰 불러오기
token_env_name = "GITHUB_TOKEN"
token_value = os.getenv(token_env_name)

if not token_value:
    raise RuntimeError(f"[GITHUB_LOGIN] 환경 변수 '{token_env_name}'가 설정되지 않았습니다. 확인해주세요.")

## 객체 생성
login_request = GithubTokenLoginRequest(token_value)
authenticator = GithubTokenAuthenticator()

In [ ]:
# 로그인
github = authenticator.githubLogin(login_request)

## 📦 데이터 수집 및 전처리
- 선택 가능 Repository 가져오기
- 내용을 가져올 Repository 선택하기
- 가져온 데이터를 chunk로 분리하여 생성

In [ ]:
# 선택 가능한 Repository 들 가져오기
# 토큰 접근 가능 레포 중, 실제로 커밋을 남긴 레포만 수집합니다.

MAX_CONCURRENT_REQUESTS = 10
_semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

def check_contributed(user, repo: Repository) -> tuple[Repository, bool]:
    """
    사용자가 해당 Repository에 커밋을 남긴 적 있는지 확인합니다.

    get_contributors() 대신 get_commits(author=)를 사용하여
    전체 기여자 목록 조회 없이 빠르게 판별합니다.

    Args:
        user: 인증된 GitHub 사용자 객체
        repo (Repository): 확인할 Repository

    Returns:
        tuple[Repository, bool]: (레포, 기여 여부)
    """
    try:
        commits = repo.get_commits(author=user.login)
        is_contributor = commits.totalCount > 0
    except GithubException as e:
        # 409 (Git Repository is empty), 404 (Not Found 등) 기대 가능한 예외만 처리하고 나머지는 전파
        if getattr(e, 'status', 0) in (404, 409):
            is_contributor = False
        else:
            raise
    return repo, is_contributor

async def fetch_contributed_repo(user, repo: Repository) -> str | None:
    """
    사용자가 기여한 Repository의 full_name을 비동기로 가져옵니다.

    Args:
        user: 인증된 GitHub 사용자 객체
        repo (Repository): 확인할 Repository

    Returns:
        str | None: 'owner/repo_name' 형식, 미기여 또는 오류 시 None
    """
    async with _semaphore:
        try:
            repo_obj, is_contributor = await asyncio.to_thread(check_contributed, user, repo)
            return repo_obj.full_name if is_contributor else None
        except GithubException as e:
            logger.error(f"[REPO_FETCH] 레포 접근 API 오류 전파 | {e}")
            raise
        except Exception as e:
            logger.error(f"[REPO_FETCH] 예상치 못한 오류 | {e}", exc_info=True)
            return None


In [ ]:
# 토큰 권한 내 모든 레포 가져오기 (조직 포함)
user = github.get_user()
repos = user.get_repos(type="all")  # public, private, fork, org 레포 포함

# 비동기 태스크 생성 및 실행
tasks = [fetch_contributed_repo(user, repo) for repo in repos]
results: List[str | None] = await asyncio.gather(*tasks)

# None 필터링 후 최종 목록 구성
my_repositories: List[str] = [name for name in results if name is not None]

logger.info(f"[REPO_FETCH] 총 {len(my_repositories)}개의 기여 Repository를 가져왔습니다.")

In [ ]:
# Repository 코드 데이터 처리
TARGET_CODE_EXTENSIONS = ('.java', '.kt', '.js', '.ts', '.md', '.py', '.yml', '.yaml', '.txt', '.xml', '.gradle', '.html', '.css', '.properties', '.dart')
MAX_FILE_SIZE = 1024 * 1024  # 1MB
EXCLUDE_DIRS = {'node_modules', 'build', 'dist', 'target', '.gradle', 'vendor', '__pycache__'}

LANGUAGE_MAP = {
    '.java': 'Java', '.kt': 'Kotlin', '.js': 'JavaScript', '.ts': 'TypeScript',
    '.md': 'Markdown', '.py': 'Python', '.yml': 'YAML',
    '.yaml': 'YAML', '.txt': 'Text', '.xml': 'XML',
    '.gradle': 'Gradle', '.html': 'HTML', '.css': 'CSS',
    '.properties': 'Properties', '.dart': 'Dart'
}

def is_test_file(file_path: str) -> bool:
    """
    테스트 파일 여부 확인

    Args:
        file_path (str): 확인하려는 파일의 경로

    Returns:
        bool: 테스트 파일이면 True, 아니면 False
    """
    path_lower = file_path.lower()
    return (
        '/test/' in path_lower or
        '/tests/' in path_lower or
        file_path.endswith('Test.java') or
        file_path.endswith('Spec.java') or
        file_path.endswith('Test.kt') or
        file_path.endswith('Spec.kt') or
        file_path.endswith('_test.py') or
        file_path.endswith('.test.js') or
        file_path.endswith('.spec.js') or
        file_path.endswith('.test.ts') or
        file_path.endswith('.spec.ts')
    )


# DTO

from dataclasses import dataclass

@dataclass
class TransformContext:
    """
    파일 변환에 필요한 컨텍스트 데이터를 담는 DTO.

    Attributes:
        repo: PyGithub Repository 객체
        item: git tree 항목 (blob)
        content_str (str): 디코딩된 파일 내용
        default_branch (str): 기본 브랜치명
        collected_at (str): 수집 날짜 문자열 (YYYY-MM-DD)
    """
    repo: object
    item: object
    content_str: str
    default_branch: str
    collected_at: str


# 추상 클래스

class MetadataTransformer(ABC):
    """
    파일 메타데이터 변환 추상 클래스.
    git tree 항목과 파일 내용을 받아 원하는 형태의 dict으로 변환합니다.
    """

    @abstractmethod
    def transform(self, context: TransformContext) -> dict:
        """
        TransformContext를 받아 메타데이터 dict으로 변환합니다.

        Args:
            context (TransformContext): 변환에 필요한 파일 컨텍스트 DTO

        Returns:
            dict: 변환된 메타데이터
        """
        pass


class RepoDataWriter(ABC):
    """
    Repository에 가져온 Metadata를 저장하기 위한 추상 클래스.
    """

    @abstractmethod
    def begin_repo(self, repo_name: str) -> None:
        """
        새로운 Repository의 데이터 저장을 시작할 때 호출됩니다.
        구현체는 이 시점에 필요한 디렉토리를 생성하거나 초기화를 수행합니다.

        Args:
            repo_name (str): Repository 이름
        """
        pass

    @abstractmethod
    def write_chunk(self, chunk: list, chunk_index: int) -> None:
        """
        청크 데이터를 저장합니다.
        저장 위치(output_dir)는 구현체가 관리합니다.

        Args:
            chunk (list): 저장할 메타데이터 리스트
            chunk_index (int): 청크 인덱스 (파일명 등에 사용)
        """
        pass


# 구현체

class JsonMetadataTransformer(MetadataTransformer):
    """
    파일 메타데이터를 JSON 직렬화 가능한 dict으로 변환하는 구현체.
    """

    def transform(self, context: TransformContext) -> dict:
        file_path = context.item.path
        extension = Path(file_path).suffix
        return {
            "repo_name": context.repo.full_name,
            "file_path": file_path,
            "file_name": Path(file_path).name,
            "extension": extension,
            "language": LANGUAGE_MAP.get(extension, 'Unknown'),
            "branch": context.default_branch,
            "content": context.content_str,
            "line_count": context.content_str.count('\n') + 1,
            "directory": str(Path(file_path).parent),
            "collected_at": context.collected_at,
            "is_test": is_test_file(file_path),
        }


class JsonChunkFileWriter(RepoDataWriter):
    """
    청크를 JSON 파일로 지정된 디렉토리에 저장하는 구현체.
    """

    def __init__(self, base_output_dir: str):
        """
        Args:
            base_output_dir (str): 파일이 저장될 루트 디렉토리
        """
        self.base_output_dir = base_output_dir
        self.current_repo_dir = ""

    def begin_repo(self, owner_name: str, repo_name: str) -> None:
        if os.path.exists(self.base_output_dir):
            shutil.rmtree(self.base_output_dir)
        self.current_repo_dir = os.path.join(self.base_output_dir, owner_name, repo_name)
        os.makedirs(self.current_repo_dir, exist_ok=True)

    def write_chunk(self, chunk: list, chunk_index: int) -> None:
        if not self.current_repo_dir:
            raise ValueError("begin_repo() must be called before write_chunk()")
            
        file_name = os.path.join(self.current_repo_dir, f"chunk_{chunk_index:03d}.json")
        with open(file_name, 'w', encoding='utf-8') as f:
            json.dump(chunk, f, ensure_ascii=False, indent=2)
        logger.info(f"[REPO_DATA] 저장: {file_name} ({len(chunk)}개 파일)")


# 비즈니스 로직

def save_repo_to_chunks(
    repo,
    transformer: MetadataTransformer,
    writer: RepoDataWriter,
    chunk_size: int = 100,
) -> None:
    """
    Repository의 파일들을 청크 단위로 수집·변환·저장합니다.

    데이터 수집(fetch)은 공통 로직으로 처리하며,
    변환(transformer)과 저장(writer)은 주입된 구현체에 위임합니다.

    last_modified는 파일별 API 추가 호출(get_commits) 대신
    트리 조회 시점의 날짜를 사용하여 Rate Limit 소모를 최소화합니다.

    Args:
        repo: PyGithub Repository 객체
        transformer (MetadataTransformer): 파일 데이터 변환 구현체
        writer (RepoDataWriter): 청크 저장 구현체
        chunk_size (int): 청크당 파일 수, 기본값은 100

    Raises:
        Exception: git tree 조회 또는 디렉토리 생성 실패 시 발생
    """
    try:
        default_branch = repo.default_branch
        tree = repo.get_git_tree(default_branch, recursive=True)
        collected_at = datetime.now().strftime('%Y-%m-%d')  # API 호출 절약: 수집 시점 날짜 사용

        logger.info(f"[REPO_DATA] 총 {len(tree.tree)}개 항목 발견 | {repo.full_name}")

        file_items = [
            item for item in tree.tree
            if item.type == "blob"
            and any(item.path.endswith(ext) for ext in TARGET_CODE_EXTENSIONS)
            and not any(excluded in Path(item.path).parts for excluded in EXCLUDE_DIRS)
            and item.size <= MAX_FILE_SIZE
        ]

        logger.info(f"[REPO_DATA] 필터링 후 {len(file_items)}개 파일 처리 예정")

        writer.begin_repo(owner_name=repo.owner.name, repo_name=repo.name)

        chunk_index = 0
        current_chunk = []

        for i, item in enumerate(file_items):
            try:
                file_content = repo.get_contents(item.path, ref=default_branch)
                content_str = file_content.decoded_content.decode('utf-8', errors='replace')

                ctx = TransformContext(
                    repo=repo,
                    item=item,
                    content_str=content_str,
                    default_branch=default_branch,
                    collected_at=collected_at,
                )
                metadata = transformer.transform(ctx)
                current_chunk.append(metadata)
                logger.info(f"[REPO_DATA] [{repo.full_name}] [{i+1}/{len(file_items)}] {item.path}")

            except GithubException as e:
                if getattr(e, 'status', 0) in (404, 422):
                    logger.warning(f"[REPO_DATA] 파일 스킵 (Not Found / Unprocessable) | {repo.full_name} | {item.path} | {e}")
                    continue
                raise
            except Exception as e:
                logger.warning(f"[REPO_DATA] 파일 스킵 (디코딩 실패 등) | {repo.full_name} | {item.path} | {e}")
                continue

            if len(current_chunk) >= chunk_size:
                writer.write_chunk(current_chunk, chunk_index)
                chunk_index += 1
                current_chunk = []

        if current_chunk:
            writer.write_chunk(current_chunk, chunk_index)
            chunk_index += 1

        logger.info(f"[REPO_DATA] 완료: {chunk_index}개 청크 저장 완료")

    except Exception as e:
        logger.error(f"[REPO_DATA] 처리 중 오류 발생 | {repo.full_name} | {e}", exc_info=True)
        raise


In [ ]:
# 번호 매겨서 Repository 목록 출력
for i, repo in enumerate(my_repositories):
    print(f"[{i:3d}] {repo}")

# 번호 입력 (쉼표로 구분, 예: 0, 2, 5)
raw = input("선택할 Repository 번호를 입력하세요 (쉼표 구분): ")

selected_repositories: List[str] = [
    my_repositories[int(i.strip())]
    for i in raw.split(",")
    if i.strip().isdigit() and int(i.strip()) < len(my_repositories)
]

logger.info(f"[REPO_SELECT] {len(selected_repositories)}개 선택됨: {selected_repositories}")

In [ ]:
json_repo_name = "data"

transformer = JsonMetadataTransformer()
writer = JsonChunkFileWriter(base_output_dir=json_repo_name)

for repo_name in selected_repositories:
    save_repo_to_chunks(
        github.get_repo(repo_name),
        transformer=transformer,
        writer=writer,
    )

## 🫙 Embedding & Store
- 생성되고 저장된 데이터를 불러가져와 임베딩하기
- 임베딩된 데이터를 DB에 저장하기

In [ ]:
# 필요 라이브러리 설치
# Voyage AI: 임베딩 퀄리티가 우수하고 코딩 특화 모델(voyage-code-2) 지원
# ChromaDB: 로컬 환경에서 무료/빠르게 벡터 DB를 구축하기 가장 적합
%pip install voyageai chromadb

In [ ]:
# Abstract & DTO
from abc import ABC, abstractmethod
from typing import List, Dict

class DataLoader(ABC):
    """데이터 소스를 읽어서 파이썬 딕셔너리(JSON) 형태의 문서 리스트로 반환하는 추상 클래스"""
    @abstractmethod
    def load(self) -> List[Dict]:
        pass

class Embedder(ABC):
    """문자열 리스트를 입력받아 임베딩 벡터 리스트로 반환하는 추상 클래스"""
    @abstractmethod
    def embed_documents(self, documents: List[str]) -> List[List[float]]:
        pass

class VectorStore(ABC):
    """문서 메타데이터, 내용, 그리고 생성된 임베딩을 바탕으로 DB에 저장하는 추상 클래스"""
    @abstractmethod
    def store(self, documents: List[Dict], embeddings: List[List[float]]) -> None:
        pass

In [ ]:
# Implementation
import voyageai
import chromadb

class JsonChunkDataLoader(DataLoader):
    def __init__(self, base_dir: str):
        self.base_dir = base_dir

    def load(self) -> List[Dict]:
        all_data = []
        if not os.path.exists(self.base_dir):
            return all_data
            
        for root, _, files in os.walk(self.base_dir):
            for file in files:
                if file.endswith('.json'):
                    file_path = os.path.join(root, file)
                    with open(file_path, 'r', encoding='utf-8') as f:
                        try:
                            chunk_data = json.load(f)
                            all_data.extend(chunk_data)
                        except json.JSONDecodeError as e:
                            logger.error(f"[DATA_LOAD] JSON 파싱 에러: {file_path} | {e}")
        return all_data

class VoyageEmbedder(Embedder):
    def __init__(self, api_key: str, model: str = "voyage-code-2"):
        # Voyage AI의 Code 모델은 코드 검색, 파악에 탁월한 성능을 보입니다.
        self.client = voyageai.Client(api_key=api_key)
        self.model = model

    def embed_documents(self, documents: List[str]) -> List[List[float]]:
        batch_size = 120  # Voyage AI 권장 최대 배치 사이즈 
        all_embeddings = []
        for i in range(0, len(documents), batch_size):
            batch = documents[i:i+batch_size]
            result = self.client.embed(batch, model=self.model)
            all_embeddings.extend(result.embeddings)
            logger.info(f"[EMBEDDING] {i + len(batch)} / {len(documents)} 벡터화 완료")
        return all_embeddings

class ChromaVectorStore(VectorStore):
    def __init__(self, persist_directory: str, collection_name: str = "github_code"):
        # 비용 효율적(무료)이고 세팅이 간편한 로컬 Chroma DB 사용
        self.client = chromadb.PersistentClient(path=persist_directory)
        self.collection = self.client.get_or_create_collection(name=collection_name)

    def store(self, documents: List[Dict], embeddings: List[List[float]]) -> None:
        if not documents or not embeddings:
            return
            
        # 레포/경로 조합으로 문서 고유 ID 생성
        ids = [f"{doc.get('repo_name', 'repo')}/{doc.get('file_path', str(i))}" for i, doc in enumerate(documents)]
        
        # ChromaDB 메타데이터 제약으로 인해 단순 string/int/bool 구조만 넘깁니다. 
        metadatas = []
        for doc in documents:
            metadatas.append({
                "repo_name": str(doc.get("repo_name", "")),
                "file_path": str(doc.get("file_path", "")),
                "file_name": str(doc.get("file_name", "")),
                "extension": str(doc.get("extension", "")),
                "language": str(doc.get("language", "")),
            })
            
        texts = [doc.get('content', '') for doc in documents]
        
        batch_size = 5000
        for i in range(0, len(ids), batch_size):
            self.collection.add(
                ids=ids[i:i+batch_size],
                embeddings=embeddings[i:i+batch_size],
                metadatas=metadatas[i:i+batch_size],
                documents=texts[i:i+batch_size]
            )
            logger.info(f"[VECTOR_DB] {i + len(ids[i:i+batch_size])} / {len(ids)} 데이터 저장 완료")

In [ ]:
# 비즈니스 로직
def process_embeddings(
    loader: DataLoader,
    embedder: Embedder,
    store: VectorStore
):
    logger.info("[EMBEDDING_PIPELINE] 데이터 로딩 시작")
    documents = loader.load()
    logger.info(f"[EMBEDDING_PIPELINE] 총 {len(documents)} 개의 문서를 불러왔습니다.")
    
    if not documents:
        logger.warning("[EMBEDDING_PIPELINE] 처리할 데이터가 없습니다.")
        return

    # 코드 임베딩 품질을 위해 메타데이터(파일 경로, 언어)를 포함한 텍스트 묶음 생성
    texts_to_embed = []
    for doc in documents:
        text = f"File: {doc.get('file_path', '')}\nLanguage: {doc.get('language', '')}\n\n{doc.get('content', '')}"
        texts_to_embed.append(text)
        
    logger.info("[EMBEDDING_PIPELINE] 임베딩 벡터 생성 진행 중... (Voyage AI)")
    embeddings = embedder.embed_documents(texts_to_embed)
    
    logger.info("[EMBEDDING_PIPELINE] 벡터 DB (Chroma)에 저장 중...")
    store.store(documents, embeddings)
    logger.info("[EMBEDDING_PIPELINE] 임베딩 파이프라인 처리 완료!")

In [ ]:
# 임베딩 후 저장
voyage_api_key = os.getenv("VOYAGE_AI_API_KEY")
if voyage_api_key:
    loader = JsonChunkDataLoader(base_dir=json_repo_name)
    embedder = VoyageEmbedder(api_key=voyage_api_key, model="voyage-code-2")
    store = ChromaVectorStore(persist_directory="./chroma_db", collection_name="github_repo_codes")
    
    process_embeddings(loader, embedder, store)
else:
    logger.warning("[EMBEDDING_PIPELINE] VOYAGE_API_KEY 환경 변수가 설정되지 않았습니다. .env에 설정 후 주석을 해제해 실행하세요.")